# task_15 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_15/data/")

In [2]:

listings = pd.read_csv(base / "listings.csv")
orders = pd.read_csv(base / "orders.csv")
refunds = pd.read_csv(base / "refunds.csv")
reviews = pd.read_csv(base / "reviews.csv")

Q1: Among sellers whose average review rating is below the median seller rating in their category but whose average delivery lag is no worse than the category median seller lag, which seller_id has the highest refund-adjusted net revenue per listing?

In [3]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["delivered_date"] = pd.to_datetime(orders["delivered_date"])
refunds["refund_date"] = pd.to_datetime(refunds["refund_date"])

merged = orders.merge(listings, on="listing_id").merge(refunds[["order_id", "refund_amount"]], on="order_id", how="left").merge(reviews, on="order_id", how="left")
merged["refund_amount"] = merged["refund_amount"].fillna(0)
merged["gross_revenue"] = merged["quantity"] * merged["price"]
merged["net_revenue"] = merged["gross_revenue"] - merged["refund_amount"]
merged["delivery_lag"] = (merged["delivered_date"] - merged["order_date"]).dt.days

seller = pd.DataFrame({
    "category": listings.groupby("seller_id")["category"].first(),
    "avg_rating": merged.groupby("seller_id")["rating"].mean(),
    "avg_lag": merged.groupby("seller_id")["delivery_lag"].mean(),
    "net_revenue_per_listing": merged.groupby("seller_id")["net_revenue"].sum() / listings.groupby("seller_id").size(),
}).reset_index()
seller["category_median_rating"] = seller.groupby("category")["avg_rating"].transform("median")
seller["category_median_lag"] = seller.groupby("category")["avg_lag"].transform("median")

eligible = seller[(seller["avg_rating"] < seller["category_median_rating"]) & (seller["avg_lag"] <= seller["category_median_lag"])]
q1 = str(eligible.sort_values(["net_revenue_per_listing", "seller_id"], ascending=[False, True]).iloc[0]["seller_id"])
print(q1)

SEL02
